# 8. Revisión del Secretariado

**Objetivo:** armar la lista de **candidatos a revisión manual** que se manda al Secretariado,
con la misma lógica del flujo que se usaba antes: primero el script de R (motor estándar) y,
sobre lo que éste dejaba sin match, `normalizar_folios` (motor refinado), que propone para
cada registro de BANCI el mejor candidato del RNPDNO.

Se comprobó contra `resultado_unificado_para_revision_30092026.xlsx`, que se armó con ese
flujo: con esta lógica coinciden 86 % de sus registros de BANCI (antes, 132 pares).

| | `reporte_diario` (indicadores) | `revision_secretariado` (este notebook) |
|---|---|---|
| Motores | en paralelo | **en cadena**: el refinado sólo ve lo que el estándar no emparejó |
| Match del estándar con fecha que difiere o FUB localizado | se anula (`regla_r_desfase_dias`) | **sigue siendo match**, como en R |
| RNPDNO contra el que busca el refinado | desaparecidos y pendientes del sexenio | **todo el RNPDNO**: localizados y todos los años |
| BANCI | no localizados | no localizados |

> Este perfil **no es para indicadores**: abre el universo del RNPDNO y quita una regla. Tiene
> su propia llave de caché, así que no pisa los resultados de `reporte_diario`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.settings import Settings
from src.pipeline import Universo
from src.pipeline.revision_sesnsp import (
    exportar_candidatos_revision, exportar_revision_fecha_hechos,
)

## 1. Configuración

El perfil está en `perfiles/revision_secretariado.toml`.

In [ ]:
cfg = Settings.desde_perfil("revision_secretariado")
print(cfg.resumen())

## 2. Matching

Si el caché ya está, esto tarda segundos. Si no, ejecuta los tres motores en cadena: el
refinado contra todo el RNPDNO tarda varios minutos.

In [ ]:
u = Universo.construir(cfg)

## 3. Candidatos a revisión manual

Un renglón por registro de BANCI sin match: el mejor candidato que propuso el refinado
(score de 1 a 99), con la columna `VALIDA` vacía para la respuesta del Secretariado. No se
vuelve a mandar lo que el Secretariado ya contestó (la regla de v6, ver
`docs/7_llaves_de_cruce.md` §7.8).

> **Pendiente:** cuando dos candidatos empatan en el score más alto, el motor se queda con
> uno sin regla explícita. Por eso el FUB propuesto puede no coincidir con el de otro flujo
> aunque el registro sea el mismo.

In [ ]:
candidatos = exportar_candidatos_revision(u)
candidatos.base.head()

## 4. Verificación de fecha de hechos

De la misma corrida: los matches que la fecha de hechos no confirma (`DIFIERE` del estándar;
definitivos del refinado con fecha fuera de ±10 días o sin fecha). En este perfil los del
estándar siguen siendo match, como en R, y se mandaban a verificar aparte (columna `VALIDO`).

In [ ]:
verificacion = exportar_revision_fecha_hechos(u)
verificacion.resumen